# Práctica Día 03 — Regularizar el MLP del Día 02

Tomamos el modelo del Día 02 y aplicamos la receta del Día 03: dropout, BatchNorm, AdamW, cosine LR, early stopping. Comparamos cuatro configuraciones.

In [ ]:
import sys; from pathlib import Path
DIA02 = Path('..').resolve().parent / 'dia_02_tema_1' / 'practica'
sys.path.insert(0, str(DIA02))
from practica_02 import load_data, prepare
from practica_03 import train_loop, metrics
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, f1_score
import pandas as pd, matplotlib.pyplot as plt
df = load_data(None)
X, y = prepare(df)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
sc = StandardScaler().fit(X_tr); X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)
lr = LogisticRegression(max_iter=2000).fit(X_tr_s, y_tr); p_lr = lr.predict_proba(X_te_s)[:,1]
yh = (p_lr>=0.5).astype(int)
lr_m = {'auc': roc_auc_score(y_te, p_lr), 'acc': accuracy_score(y_te, yh),
        'recall': recall_score(y_te, yh), 'f1': f1_score(y_te, yh)}
lr_m

In [ ]:
configs = [{'name': 'base', 'p': 0.0, 'wd': 0.0},
           {'name': '+drop', 'p': 0.3, 'wd': 0.0},
           {'name': '+drop+L2', 'p': 0.3, 'wd': 1e-4},
           {'name': 'todo+cos', 'p': 0.3, 'wd': 1e-4}]
results = {'LR': lr_m}; hists = {}
for c in configs:
    m, h = train_loop(X_tr_s, y_tr, X_te_s, y_te, p=c['p'], wd=c['wd'], epochs=80)
    results[c['name']] = metrics(m, X_te_s, y_te); hists[c['name']] = h
pd.DataFrame(results).T

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 6))
for n, h in hists.items():
    ax[0,0].plot(h['train'], label=n); ax[0,1].plot(h['val'], label=n)
    ax[1,0].plot(h['lr'], label=n)
ax[0,0].set_title('loss/train'); ax[0,1].set_title('loss/val')
ax[1,0].set_title('optim/lr'); ax[1,0].set_yscale('log')
ax[0,0].legend(fontsize=8); plt.tight_layout(); plt.show()

## Análisis (sin IA)

1. ¿Qué componente aportó más a la AUC?
2. ¿Qué componente redujo más el overfitting?
3. ¿Cuál fue el coste en épocas/segundos?
4. Si tuvieras que defender una sola configuración delante del cliente, ¿cuál y por qué?

## Declaración de uso de IA

| Sección | Herramienta | Para qué |
|---|---|---|
| Código | … | … |
| Análisis | Ninguna | — |